<a href="https://colab.research.google.com/github/vikrant48/AI-ML-python-code/blob/main/LangGraph.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
!pip install langchain langchain-community langchain-huggingface langchain-text-splitters langchain-core langchain-groq langgraph
!pip install chromadb sentence-transformers pypdf rank-bm25 ragas datasets

In [ ]:
import os
from google.colab import userdata

os.environ["GROQ_API_KEY"] = userdata.get("GROQ_API_KEY").strip()
os.environ["LANGCHAIN_API_KEY"] = userdata.get("LANGCHAIN_API_KEY")
os.environ["LANGCHAIN_TRACING_V2"]= userdata.get("LANGCHAIN_TRACING_V2")
os.environ["ANONYMIZED_TELEMETRY"] = "False"

In [ ]:
# create chunks

from langchain_text_splitters import RecursiveCharacterTextSplitter
splitter = RecursiveCharacterTextSplitter(
    chunk_size=400,
    chunk_overlap=100
)

In [ ]:
# for embedding
from langchain_huggingface import HuggingFaceEmbeddings

embeddings = HuggingFaceEmbeddings(
    model_name="BAAI/bge-small-en"   # better for retrieval than MiniLM
)

In [ ]:
#for LLM
from langchain_groq import ChatGroq

llm = ChatGroq(
    model="llama-3.1-8b-instant",
    temperature=0.2
)

In [ ]:
import re

def extract_name(text):
    lines = text.split("\n")

    for line in lines[:5]:  # check first 5 lines
        line = line.strip()

        # simple rule: name = line with only alphabets and spaces
        if re.match(r"^[A-Za-z ]{3,}$", line):
            return line

    return "Unknown"

In [ ]:
# here for multiple resume (load pdf and chunking )
from langchain_community.document_loaders import PyPDFLoader

resume_files = [
    "resume1.pdf",
    "resume2.pdf",
    "resume3.pdf"
]
# for store chunking
chunks = []

for file in resume_files:
    loader = PyPDFLoader(file)
    docs = loader.load()

    if not docs:
        print(f"Skipping empty file: {file}")
        continue

    first_page_text = docs[0].page_content
    name = extract_name(first_page_text)
    print("Extracted Name:", name)

    for doc in docs:
        doc.metadata["name"] = name
        doc.metadata["source"] = file   # useful later

    file_chunk = splitter.split_documents(docs)

    chunks.extend(file_chunk)


In [ ]:
#create and store
from langchain_community.vectorstores import Chroma

vectorstore = Chroma.from_documents(
    documents=chunks,
    embedding=embeddings,
    persist_directory="./chroma_db"
)

print("Vector DB created")

In [ ]:
#Create BM25
from rank_bm25 import BM25Okapi

corpus_texts = [chunk.page_content for chunk in chunks]
corpus_tokens = [text.lower().split() for text in corpus_texts]

bm25 = BM25Okapi(corpus_tokens)

print("BM25 ready")

In [ ]:
# hybrid search
import numpy as np
def get_dense_rankings(query, k=10):

    dense_docs = vectorstore.similarity_search(query, k=k)

    ranked_indices = []

    for doc in dense_docs:

        for i, chunk in enumerate(chunks):

            if chunk.page_content == doc.page_content:
                ranked_indices.append(i)
                break

    return ranked_indices


def get_bm25_rankings(query, k=10):

    scores = bm25.get_scores(query.lower().split())

    ranked_indices = np.argsort(scores)[::-1][:k]

    return ranked_indices.tolist()


def reciprocal_rank_fusion(rank_lists, k=60):

    scores = {}

    for rank_list in rank_lists:

        for rank, idx in enumerate(rank_list):

            scores[idx] = scores.get(idx, 0) + 1 / (k + rank + 1)

    return sorted(
        scores.items(),
        key=lambda x: x[1],
        reverse=True
    )

In [ ]:
def hybrid_search(query, k=10):

    dense_rank = get_dense_rankings(query, k=10)

    bm25_rank = get_bm25_rankings(query, k=10)

    fused = reciprocal_rank_fusion([
        dense_rank,
        bm25_rank
    ])

    results = []
    seen = set()

    for idx, score in fused:

        if idx not in seen:

            results.append((chunks[idx], score))

            seen.add(idx)

        if len(results) >= k:
            break

    return results

In [ ]:
#RERANKER
from sentence_transformers import CrossEncoder
reranker = CrossEncoder(
    "cross-encoder/ms-marco-MiniLM-L-6-v2"
)

In [ ]:
def rerank_results(query, results, top_n=5):

    docs = [doc for doc, _ in results]

    pairs = [
        (query, doc.page_content)
        for doc in docs
    ]

    scores = reranker.predict(pairs)

    ranked = sorted(
        zip(scores, docs),
        key=lambda x: x[0],
        reverse=True
    )

    return ranked[:top_n]

In [ ]:
#LANGGRAPH STATE
from typing import TypedDict, List
class ResumeState(TypedDict):

    question: str

    retrieved_docs: list

    reranked_docs: list

    final_answer: str

In [ ]:
#RETRIEVE NODE
def retrieve_node(state: ResumeState):

    results = hybrid_search(
        state["question"],
        k=10
    )

    state["retrieved_docs"] = results

    return state

In [ ]:
#RERANK NODE
def rerank_node(state: ResumeState):

    reranked = rerank_results(
        state["question"],
        state["retrieved_docs"],
        top_n=5
    )

    state["reranked_docs"] = reranked

    return state


In [ ]:
#ANSWER NODE
def answer_node(state: ResumeState):

    context_parts = []

    for i, (score, doc) in enumerate(state["reranked_docs"]):

        meta = doc.metadata

        context_parts.append(f"""
Candidate {i+1}

Source: {meta.get('source')}
Experience: {meta.get('years')} years

Relevance Score: {round(float(score), 4)}

Resume:
{doc.page_content[:700]}
""")

    context = "\n\n-----------------\n\n".join(context_parts)

    response = llm.invoke(f"""
You are an AI hiring assistant.

Job Requirement:
{state["question"]}

Candidates:
{context}

Instructions:
1. Rank top candidates
2. Explain why they match
3. Mention key skills
4. Give final recommendation
""")

    state["final_answer"] = response.content

    return state

In [ ]:
#BUILD LANGGRAPH
from langgraph.graph import StateGraph, END
graph = StateGraph(ResumeState)

graph.add_node("retrieve", retrieve_node)

graph.add_node("rerank", rerank_node)

graph.add_node("answer", answer_node)

# Edges
graph.add_edge("retrieve", "rerank")

graph.add_edge("rerank", "answer")

graph.add_edge("answer", END)

# Entry point
graph.set_entry_point("retrieve")

# Compile app
app = graph.compile()

print("LangGraph app ready")

In [ ]:
#RUN QUERY
query = """
Looking for a Java backend engineer with:
- Spring Boot
- AWS
- REST APIs
- Microservices
- SQL knowledge
"""

result = app.invoke({
    "question": query,
    "retrieved_docs": [],
    "reranked_docs": [],
    "final_answer": ""
})

In [ ]:
print(result["final_answer"])